# Nested derivatives

`DC`, `PartialD`, and `FS` can be nested. The compiler expands the inner
operator first and then applies the outer one with the product rule.


## Setup


In [1]:
import re
import sys
from fractions import Fraction
from pathlib import Path

from symbolica import Expression, S

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ANSI_ESCAPE_RE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")


def clean(text):
    return ANSI_ESCAPE_RE.sub("", str(text))


def show(title, result):
    print("==========")
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", clean(expression))
            print()
    else:
        print(clean(result))
        print()


def show_model(model, *fields, compact_form=None, sum_notation=None, simplify=True):
    source_terms = model.lagrangian_decl.source_terms
    if source_terms:
        lagrangian_source = (
            sum(source_terms[1:], source_terms[0])
            if len(source_terms) > 1
            else source_terms[0]
        )
        show("Lagrangian", lagrangian_source)
    lagrangian = model.lagrangian()
    if fields:
        show("Feynman Rule", lagrangian.feynman_rule(*fields, include_delta=False, simplify=simplify))
    else:
        show("Feynman Rules", lagrangian.feynman_rule(include_delta=False, simplify=simplify))
    if compact_form is not None:
        show("Compact Form", compact_form)
    if sum_notation is not None:
        show("Sum Notation", sum_notation)

from feynpy import (
    COLOR_ADJ_INDEX,
    COLOR_FUND_INDEX,
    DC,
    FS,
    Field,
    GaugeGroup,
    GaugeRepresentation,
    LORENTZ_INDEX,
    Model,
    PartialD,
)
from symbolic.spenso_structures import gauge_generator, structure_constant


## Nested covariant derivatives on matter

`DC(DC(Phi, mu), mu)` transforms like `Phi`. The outer covariant derivative
hits every expanded inner branch and adds the U(1) gauge branch.


In [2]:
mu, nu, rho, a = S("mu"), S("nu"), S("rho"), S("a")
g, q = S("g"), S("q")

photon = Field("A", spin=1, self_conjugate=True, symbol=S("A"), indices=(LORENTZ_INDEX,))
scalar = Field(
    "Phi",
    spin=0,
    self_conjugate=False,
    symbol=S("Phi"),
    conjugate_symbol=S("Phibar"),
    quantum_numbers={"Q": q},
)
u1 = GaugeGroup("U1", abelian=True, coupling=g, gauge_boson=photon, charge="Q")

nested_scalar = Model(
    scalar.bar * DC(DC(scalar, mu), mu),
    gauge_groups=(u1,),
    fields=(photon, scalar),
)
show_model(nested_scalar)


Lagrangian
Phi.bar * DC(DC(Phi, mu), mu)

Feynman Rules
3 vertex signature(s)

Vertex: ('Phi.bar', 'Phi')
Rule: -1𝑖*pcomp(q2,mu1_int)^2

Vertex: ('Phi.bar', 'Phi', 'A')
Rule: -2𝑖*g*q*pcomp(q2,mu3)-1𝑖*g*q*pcomp(q3,mu3)

Vertex: ('Phi.bar', 'Phi', 'A', 'A')
Rule: -2𝑖*g^2*q^2*g(mink(4, mu3),mink(4, mu4))



## Derivatives of field strengths

`PartialD(FS(...), mu)` expands the field strength and then differentiates
each gauge field. `DC(FS(...), mu)` treats the field strength as adjoint-valued.


In [3]:
gluon = Field(
    "G",
    spin=1,
    self_conjugate=True,
    symbol=S("G"),
    indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX),
)
su3 = GaugeGroup(
    "SU3",
    abelian=False,
    coupling=g,
    gauge_boson=gluon,
    structure_constant=structure_constant,
    representations=(
        GaugeRepresentation(index=COLOR_FUND_INDEX, generator_builder=gauge_generator, name="fund"),
    ),
)

partial_fs = Model(
    gluon(nu, a) * PartialD(FS(su3, mu, nu, a), mu),
    gauge_groups=(su3,),
    fields=(gluon,),
)
show_model(partial_fs, simplify=False)

covariant_fs = Model(
    gluon(nu, a) * DC(FS(su3, mu, nu, a), mu),
    gauge_groups=(su3,),
    fields=(gluon,),
)
show_model(covariant_fs, simplify=False)

double_covariant_fs = Model(
    gluon(nu, a) * DC(DC(FS(su3, mu, nu, a), rho), rho),
    gauge_groups=(su3,),
    fields=(gluon,),
)
show_model(double_covariant_fs, simplify=False)


Lagrangian
G * PartialD(FS(SU3, mu, nu, a), mu)

Feynman Rules
2 vertex signature(s)

Vertex: ('G', 'G')
Rule: 1𝑖*(-g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1_int)^2*delta(G,G)^2-g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*pcomp(q2,mu1_int)^2*delta(G,G)^2)+1𝑖*(g(mink(4, mu1_int),mink(4, mu1))*g(mink(4, mu2),mink(4, mu2_int))*g(coad(8, a1),coad(8, a2))*pcomp(q2,mu1_int)*pcomp(q2,mu2_int)*delta(G,G)^2+g(mink(4, mu1_int),mink(4, mu2))*g(mink(4, mu1),mink(4, mu2_int))*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1_int)*pcomp(q1,mu2_int)*delta(G,G)^2)

Vertex: ('G', 'G', 'G')
Rule: 1𝑖*(-1𝑖*g*g(mink(4, mu1_int),mink(4, mu3))*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a1),coad(8, a3),coad(8, a2))*pcomp(q2,mu1_int)*delta(G,G)^3-1𝑖*g*g(mink(4, mu1_int),mink(4, mu3))*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a2),coad(8, a3),coad(8, a1))*pcomp(q1,mu1_int)*delta(G,G)^3-1𝑖*g*g(mink(4, mu1_int),mink(4, mu1))*g(mink(4, mu3),mink(4, mu2))*f(coad(8, a2),coad(8, a1),coad(8, a3))*